# MCP 04 · 权限：JWT 认证（签发 → 校验 → 携带令牌调用）

前面几课把「服务端 ↔ 客户端」打通了，但那些服务都是**裸奔**的——谁连上都能调工具。
这一课给它加一把锁：**JWT（JSON Web Token）认证**。核心是拆成三个角色互相制衡：

| 角色 | 职责 | 本课的代码形态 |
|---|---|---|
| 认证服务（外部系统） | 校验口令 → **签发**令牌，定义用户身份（`sub`）与权限（`scope`） | `FastAPI` 的 `/login` 接口（端口 9110） |
| 服务端 | **验证**令牌签名 → 提取声明 → 决定放行 | `FastMCP(..., auth=JWTVerifier(...))`（端口 8140） |
| 客户端 | **持有**令牌，请求时放进 `Authorization` 头，**无权**自己改权限 | `login()` 拿令牌 + `BearerAuth(token)` 连接 |

为什么必须拆成三个角色？如果 MCP 服务端自己签发令牌，「谁能用哪个工具」就由服务端自己说了算，
客户端连上服务端就能给自己发通行证——**权限形同虚设**。拆开后：签发权在认证服务、校验权在服务端、
持有权在客户端，三者互相制衡。

JWT 的结构是 `header.payload.signature`（三段 Base64URL，用 `.` 分隔）：

| 段 | 内容 | 说明 |
|---|---|---|
| header | `{"alg": "HS256", "typ": "JWT"}` | 签名算法 |
| payload | `{"sub": 用户, "scope": 权限, "exp": 过期, "iat": 签发}` | 声明；**只是 Base64 编码，不是加密** |
| signature | `HMAC-SHA256(header.payload, SECRET_KEY)` | 改一个字签名就对不上 |

> **本 notebook 由 `Agent/_py_source/05_mcp/` 下 6 个脚本合并而成**（**编号错位**，见下表）：
>
> | 源文件 | 行数 | 角色 |
> |---|---|---|
> | `07_权限_认证服务.py` ↔ `08_权限_认证服务_jxsd.py` | 83 / 267 | 认证服务（签发 JWT） |
> | `08_权限_服务端.py` ↔ `09_权限_服务端_jxsd.py` | 55 / 233 | MCP 服务端（JWTVerifier 校验） |
> | `09_权限_客户端.py` ↔ `10_权限_客户端_jxsd.py` | 53 / 300 | 客户端（携带 Bearer Token） |

**官方文档**
- FastMCP Auth（JWTVerifier）：<https://gofastmcp.com/deployment/auth>
- MCP 规范（授权）：<https://modelcontextprotocol.io/specification/2025-06-18/basic/authorization>
- PyJWT：<https://pyjwt.readthedocs.io/>

## 运行条件

| 项 | 说明 |
|---|---|
| 🔴 运行档位 | **需外部服务** —— notebook 内**自己把两个服务都起起来**（认证服务 9110 + MCP 服务端 8140），末尾自动关闭 |
| 依赖 | `fastapi` / `uvicorn` / `pyjwt` / `fastmcp` / `httpx` / `requests`（本项目 venv 已装） |
| 密钥 | 无外部密钥 —— 用内置演示密钥（HS256 对称签名），认证服务与服务端**共享同一把** |
| 前置服务 | 无（两个服务都由 notebook 自己拉起，不必另开窗口） |
| 端口 | **9110**（认证服务）+ **8140**（MCP 服务端）；同章 5 个 notebook 并发，各占各的，别用 8000/9000 |
| 预计耗时 | 约 15~30 秒 |

> 两个服务都是**常驻服务**，直接 `uvicorn.run()` / `mcp.run()` 会永久阻塞内核。所以本 notebook
> 走模板第 6 节第 5 条：把两个服务各自写成能独立跑的 `auth_service.py` / `mcp_server.py`，
> 用 `subprocess.Popen` 后台起进程、轮询端口就绪（再真发一个请求做**语义校验**），
> **最后一个 cell** 用 `taskkill /F /T` 连子进程树一起收掉。

## 本节地图

一次完整的带认证调用，四步链路：

```mermaid
graph LR
    C["客户端"] -->|"① POST /login<br/>用户名+口令"| A["认证服务 :9110<br/>FastAPI"]
    A -->|"② 返回 JWT<br/>sub / scope / exp"| C
    C -->|"③ Authorization: Bearer JWT"| S["MCP 服务端 :8140<br/>FastMCP + JWTVerifier"]
    S -->|"④ 验签通过 → 工具结果"| C
```

上面这张图等价于下面这张表（**裸 JupyterLab 不渲染 mermaid，看表即可**）：

| 步骤 | 谁 → 谁 | 携带什么 | 谁校验 |
|---|---|---|---|
| ① 登录 | 客户端 → 认证服务 | `{"username", "password"}` | 认证服务查用户库，对口令 |
| ② 签发 | 认证服务 → 客户端 | `access_token`（JWT） | — |
| ③ 调用 | 客户端 → MCP 服务端 | `Authorization: Bearer <JWT>` | 服务端验签 + 验时间 + 验 scope |
| ④ 结果 | MCP 服务端 → 客户端 | 工具返回值 | — |

与上一节的衔接：`03_三种Agent调用.ipynb` 的服务端**没上锁**；这一课给它加上 `JWTVerifier`，
客户端就多了一步「先登录拿令牌」，其余调用方式不变。

与下一节的衔接：`05_部署与调试.ipynb` 讲怎么把这类服务部署成生产环境（Docker / uvicorn 多进程）。

## 0. 环境引导

notebook 的**工作目录默认是它自己所在的文件夹**，而本项目代码都写
`from config import settings`（`config.py` 在仓库根）。所以每个 notebook 的第一格
统一做一件事：**向上找到仓库根，切过去，并塞进 `sys.path`**。

> 本课其实用不到 `config`（两个服务都是本地演示、不调大模型），但这一格仍保留——保持全仓统一，
> 并给出 `NB_DIR` / `WORKDIR`：本课要把两个服务脚本落盘到 `WORKDIR` 下的专属子目录，靠的就是它们。

In [ ]:
# ===== 环境引导（每个 notebook 的第一格，不要改）=====
import os
import sys
from pathlib import Path

NB_DIR = Path.cwd()                 # notebook 所在目录（chdir 之前先抓住）
ROOT = NB_DIR
while not (ROOT / "config.py").exists():
    if ROOT.parent == ROOT:
        raise RuntimeError("没找到 config.py：请在 Python_Base 仓库内运行本 notebook")
    ROOT = ROOT.parent

os.chdir(ROOT)                      # 让相对路径（data/、output.txt 等）都相对仓库根
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

WORKDIR = NB_DIR / "tmp_nb_work"    # 本 notebook 的临时工作目录（已被 .gitignore 覆盖）
WORKDIR.mkdir(exist_ok=True)

print("仓库根：", ROOT)
print("临时目录：", WORKDIR)

## 0.1 前置条件自检

四件事，缺一件后面都会以报错收场，所以先查一遍：

1. **`NO_PROXY` 必须含 `127.0.0.1`** —— 本机 Clash 会把回环请求拦走，表现为「服务起来了却连接被拒」；
2. **六个依赖包**是否都在（`fastapi` / `uvicorn` / `jwt` / `fastmcp` / `httpx` / `requests`）；
3. **端口 9110 / 8140 是否空闲**（同章并发，撞了会连到别人的服务）。

In [ ]:
import importlib.util
import socket as _socket

# 本机 Clash 拦回环：把 127.0.0.1 补进 NO_PROXY（已设过就不动）
for _key in ("NO_PROXY", "no_proxy"):
    _val = os.environ.get(_key, "")
    if "127.0.0.1" not in _val:
        os.environ[_key] = (_val + "," if _val else "") + "127.0.0.1,localhost"

NEEDED = ["fastapi", "uvicorn", "jwt", "fastmcp", "httpx", "requests"]
_missing = [name for name in NEEDED if importlib.util.find_spec(name) is None]
print("NO_PROXY =", os.environ.get("NO_PROXY"))
if _missing:
    print("[跳过] 缺少依赖：", _missing)
    print("   安装命令： uv add " + " ".join(_missing))
else:
    print("✅ 依赖齐全：", NEEDED)


def _port_in_use(host: str, port: int) -> bool:
    """探测端口是否已被监听。短超时，别让探测把 notebook 拖慢。"""
    with _socket.socket() as s:
        s.settimeout(0.5)
        return s.connect_ex((host, port)) == 0


NB_AUTH_PORT = 9110     # 认证服务（签发 JWT）——本 notebook 独占端口
NB_MCP_PORT = 8140      # MCP 服务端（校验 JWT）——本 notebook 独占端口
if _port_in_use("127.0.0.1", NB_AUTH_PORT):
    print(f"[跳过] 端口 {NB_AUTH_PORT} 已被占用，请先释放（本 notebook 需要 {NB_AUTH_PORT}）")
elif _port_in_use("127.0.0.1", NB_MCP_PORT):
    print(f"[跳过] 端口 {NB_MCP_PORT} 已被占用，请先释放（本 notebook 需要 {NB_MCP_PORT}）")
else:
    print(f"✅ 端口 {NB_AUTH_PORT} / {NB_MCP_PORT} 空闲，可以起服务。")

## 1. 课案原版：三个文件的最短形态

课案原版三个文件各只有几十行，正好够「签发 / 校验 / 调用」各出现一次。
**先看最短实现，再看完整版**，两者的差距就是本课要讲的全部内容。

> 原版把「运行方式」写死在文件头：`uv run 07_权限_认证服务.py serve`（起 9000 端口）、
> `uv run 08_权限_服务端.py http`（起 8000 端口）、`uv run 09_权限_客户端.py`（连 8000）。
> 这些端口在 notebook 里会撞同章其它课，所以**只作对照保留，不真跑**（完整版 §2 会用 9110 / 8140 真跑）。

### 1.1 认证服务：签发 / 校验 JWT（`07_权限_认证服务.py`）

最小认证服务只有两件事：`create_token()` 把 `sub`（用户）+ `scope`（权限）+ `iat`/`exp`（时间）
签进 payload，`verify_token()` 验签 + 验过期。

注意 `scope` 是**空格分隔的字符串**（`" ".join(scopes)`）——这是 MCP 官方规范，不是数组，也是最容易写错的一处。

In [ ]:
# ---------- 1.1 认证服务原版：签发 / 校验 JWT ----------
from datetime import datetime, timedelta, timezone

import jwt
from fastapi import FastAPI
from pydantic import BaseModel

# ============ 密钥配置 ============
# 生产环境：私钥自己保管（签发用），公钥公开（校验用）
# 演示环境：直接用对称密钥（同一个字符串既签发又校验）
JWT_SECRET = "demo-secret-change-me"   # 生产环境必须换成 RSA 私钥/强随机密钥
ALGORITHM = "HS256"
TOKEN_EXPIRE_MINUTES = 60


def create_token(user_id: str, scopes: list[str]) -> str:
    """签发 JWT：把用户身份和权限写进 payload"""
    payload = {
        "sub": user_id,                     # subject：用户唯一标识
        "scope": " ".join(scopes),          # 权限列表（如 read write execute）
        "iat": datetime.now(timezone.utc),  # 签发时间
        "exp": datetime.now(timezone.utc) + timedelta(minutes=TOKEN_EXPIRE_MINUTES),
    }
    return jwt.encode(payload, JWT_SECRET, algorithm=ALGORITHM)


def verify_token(token: str) -> dict:
    """校验 JWT：验签 + 验过期，返回 payload；不合法直接抛异常"""
    return jwt.decode(token, JWT_SECRET, algorithms=[ALGORITHM])


# ---------- 作为 HTTP 登录接口运行 ----------
app = FastAPI(title="MCP 认证服务")


class LoginRequest(BaseModel):
    username: str
    password: str


@app.post("/login")
def login(req: LoginRequest):
    """
    演示版：任意用户名 + 密码 123456 都能登录。
    生产版：查数据库校验密码（密码哈希存储）。
    """
    if req.password != "123456":
        return {"error": "用户名或密码错误"}

    # 按用户角色分配权限
    scopes = ["read", "execute"] if req.username == "admin" else ["read"]
    return {"access_token": create_token(req.username, scopes), "token_type": "Bearer"}

下面是原版文件头里那段「命令行入口」的**原样保留**（`serve` 起 HTTP 服务 / 默认直接签发一个演示令牌）。
notebook 里**不调用** `_cli_main()` —— `uvicorn.run()` 是常驻服务，直接跑会永久阻塞内核；
但「默认直接签发演示令牌」这一支是离线可跑的，我们顶格跑一遍拿真实输出。

In [ ]:
# ---------- 1.1（附）原版的命令行入口 + 离线跑「签发 / 校验」这一支 ----------
def _cli_main():
    if len(sys.argv) > 1 and sys.argv[1] == "serve":
        # 以 HTTP 服务运行登录接口（notebook 里不调用：会阻塞内核）
        import uvicorn
        uvicorn.run(app, host="127.0.0.1", port=9000)
    else:
        token = create_token("user-1001", ["read", "execute"])
        print("签发的 JWT：", token)
        print("校验结果：", verify_token(token))

# 顶格直接跑「签发 + 校验」（等价于上面 else 分支，不经过 sys.argv）
token = create_token("user-1001", ["read", "execute"])
print("签发的 JWT：", token)
print("校验结果：", verify_token(token))

### 预期输出

```text
签发的 JWT： eyJhbGciOiJIUzI1NiIsInR5cCI6IkpXVCJ9.<这里每次运行都不同，且是敏感值，已省略>...（共 NNN 字符）
校验结果： {'sub': 'user-1001', 'scope': 'read execute', 'iat': 1750000000, 'exp': 1750003600}
```

> ⚠️ **本格输出不确定**：`iat` / `exp` 是**时间戳**，签名段也是**每次运行都不同**；
> 上面的 `NNN` 与数字都只是示意，**别逐字比对**。真正要确认的是三点：

1. 第一行的三段式 `header.payload.signature` 用 `.` 分隔 —— 这就是 JWT 的样子；
2. `verify_token` 能解出 payload，且 `sub` / `scope` 就是签发时写进去的值；
3. 令牌里的 `scope` 是**空格分隔字符串** `"read execute"`，不是数组；
4. 旁边那条 `InsecureKeyLengthWarning` 是 pyjwt 的善意提醒：原版演示密钥
   `demo-secret-change-me` 只有 21 字节，低于 HS256 建议的 32 字节 ——
   完整版 §2.1 会换成 40 字节的密钥，警告就消失了。

### 1.2 带认证的 MCP 服务端（`08_权限_服务端.py`）

给 MCP 服务加上 `JWTVerifier`：`auth=verifier` 就是课案说的「校验 Bearer Token 的中间件」——
客户端请求头里没有合法 `Authorization: Bearer <token>` 就直接 401，**根本进不到工具函数**，
所以业务代码里一行鉴权都不用写。

> 注意 `required_scopes=["read"]`：所有调用至少要带 `read` 权限（来自令牌 payload 的 `scope` 声明）。

In [ ]:
# ---------- 1.2 服务端原版：JWTVerifier 校验 Bearer Token ----------
from fastmcp import FastMCP
from fastmcp.server.auth.providers.jwt import JWTVerifier

# 与认证服务共享的密钥（生产环境换成公钥/JWKS）
JWT_SECRET = "demo-secret-change-me"

# 创建校验器：要求请求头 Authorization: Bearer <token>
# 演示用对称密钥（HS256）：public_key 参数即共享密钥
verifier = JWTVerifier(
    public_key=JWT_SECRET,
    algorithm="HS256",
    required_scopes=["read"],  # 所有调用至少要有 read 权限
)

mcp = FastMCP(name="受保护的MCP服务", auth=verifier)


@mcp.tool
def get_weather(city: str) -> str:
    """查询指定城市的实时天气"""
    weather_map = {"上海": "晴 25 度", "北京": "多云 18 度"}
    return weather_map.get(city, f"{city} 天气未知")


@mcp.tool
def admin_reset() -> str:
    """重置系统（危险操作，需要 write 权限才能调用）"""
    # 细粒度权限：在函数内自己校验 scope（FastMCP 会把 Token 信息放进上下文）
    return "系统已重置"

原版文件头的运行方式：`uv run 08_权限_服务端.py http`。带认证的服务**只能以 HTTP 运行**——
stdio 模式没有 HTTP 请求头，塞不进 `Authorization`，中间件拿不到令牌，所有调用都会 401。
下面这句 `mcp.run(...)` 原样保留在 `_cli_main()` 里，notebook **不调用**（会阻塞内核），
完整版 §2 会用 `subprocess.Popen` 真起这个服务（端口 8140）。

In [ ]:
def _cli_main():
    # 带认证的服务只能以 HTTP 运行（stdio 无法传 Token 头）
    mcp.run(transport="http", host="127.0.0.1", port=8000, path="/mcp")

### 1.3 带认证的客户端（`09_权限_客户端.py`）

客户端三步：`login()` 找认证服务换令牌 → `BearerAuth(token)` 把令牌塞进请求头 → `Client` 连 MCP 服务。
原版把地址写死成 `9000`（认证服务）和 `8000`（MCP 服务），notebook 里只作对照、不真跑（§2 改用 9110/8140）。

In [ ]:
# ---------- 1.3 客户端原版：login 拿令牌 + BearerAuth 连接 ----------
import asyncio

import httpx
from fastmcp import Client
from fastmcp.client.auth import BearerAuth
from fastmcp.client.transports import StreamableHttpTransport

MCP_URL = "http://127.0.0.1:8000/mcp"


def login(username: str, password: str) -> str:
    """向认证服务换取 JWT"""
    resp = httpx.post(
        "http://127.0.0.1:9000/login",
        json={"username": username, "password": password},
    )
    resp.raise_for_status()
    return resp.json()["access_token"]


async def main():
    # 1. 登录拿令牌
    token = login("admin", "123456")
    print("拿到 JWT：", token[:40], "...")

    # 2. 带 Bearer Token 连接 MCP 服务（fastmcp 3.x 客户端必须异步使用）
    transport = StreamableHttpTransport(url=MCP_URL, auth=BearerAuth(token))
    async with Client(transport) as client:
        tools = await client.list_tools()
        print("工具列表：", [t.name for t in tools])

        result = await client.call_tool("get_weather", {"city": "上海"})
        print("调用结果：", result)

原版文件头：`uv run 09_权限_客户端.py` → 末尾是 `asyncio.run(main())`。notebook 里**不这样跑**——
ipykernel 主线程已经在事件循环里，顶层 `asyncio.run()` 会抛 `RuntimeError`（见「常见坑」第 2 条）。
所以这句原样保留在 `_cli_main()` 里作对照，真跑时用 §2.5 的 `run_async()` 把它丢进新线程。

In [ ]:
def _cli_main():
    asyncio.run(main())

## 2. 完整版：自起认证服务 + 自起 MCP 服务端 + 完整链路

课案原版把三个文件各自跑在三个窗口里，还要手动按顺序起服务。完整版把它们**合进一条链路**，
并且用真实服务 + 真实令牌把「谁说了算」演示清楚。下面分五步：

1. **认证服务完整版**：共享密钥 + 模拟用户库 + 签发/校验/登录（§2.1）；
2. **MCP 服务端完整版**：`JWTVerifier` 校验器 + 两个工具（§2.2）；
3. **客户端完整版**：`login` + `try_connect` + 正例与三个反例（§2.3）；
4. **起服务**：把两个服务写成独立脚本，`subprocess.Popen` 后台起 + 轮询 + 语义校验（§2.4）；
5. **跑链路** + **收尾**（§2.5 / §2.6）。

> 原脚本（`_jxsd`）的「自起服务」用的是**线程内 uvicorn + importlib 加载兄弟文件**那一套；
> 本 notebook 改用 **`subprocess.Popen` 独立进程**（模板第 6 节第 5 条）——
> 原因见「常见坑」第 4、6 条。原脚本那些 `load_sibling` / `start_*_in_thread` 会原样保留在下面各小节里作对照，但**不调用**。

### 2.1 认证服务完整版：共享密钥 + 模拟用户库 + 签发/校验/登录（`08_权限_认证服务_jxsd.py`）

和原版最大的差别：密钥从环境变量读（**不硬编码进代码**），用户库换成了 `USERS` 字典，
`scope` 直接写成空格分隔字符串。这段是完整版认证服务的**核心**，后面 §2.4 会把它原样落盘成独立脚本起起来。

In [ ]:
# ---------- 2.1 认证服务完整版核心：密钥 / 用户库 / 签发 / 校验 / 登录 ----------
import datetime
import os

import jwt
from fastapi import FastAPI
from pydantic import BaseModel

# ================================================================
# 一、共享密钥
# ================================================================
# ⚠️ 绝不能把密钥硬编码进代码 —— 提交进仓库就等于把签发权公开了。
#    正确做法：从环境变量读，本地开发用一份演示默认值兜底。
SECRET_KEY_ENV = "MCP_JWT_SECRET"
SECRET_KEY = os.environ.get(SECRET_KEY_ENV) or "demo-shared-secret-key-minimum-32-chars"
SECRET_KEY_FROM_ENV = bool(os.environ.get(SECRET_KEY_ENV))

# HS256 = HMAC-SHA256，对称签名：签发和校验用同一个密钥。
ALGORITHM = "HS256"
TOKEN_EXPIRE_HOURS = 1     # 课案：过期时间 1 小时后令牌失效

app = FastAPI(title="MCP 认证服务")


# ================================================================
# 二、模拟用户数据库
# ================================================================
# 真实项目里**永远不要明文存口令**，要存 bcrypt/argon2 的哈希值。
USERS = {
    "alice": {"password": "pass123"},
    "bob":   {"password": "pass456"},
}


class LoginRequest(BaseModel):
    """登录请求体：FastAPI 会自动把 JSON 反序列化成这个模型并做类型校验"""
    username: str
    password: str


# ================================================================
# 三、签发 JWT
# ================================================================
def create_token(user_id: str, scopes: str = "read execute") -> str:
    """签发 JWT 令牌

    这里补一个 scope 字段，原因是：§2.2 里 `JWTVerifier(required_scopes=["read"])`
    是靠 payload 里的 **scope 声明**做权限判断的。没有 scope，服务端只校验签名，
    无法区分 read / write 权限。
    """
    now = datetime.datetime.now(datetime.timezone.utc)
    payload = {
        "sub": user_id,                                          # subject：用户唯一标识
        "scope": scopes,                                         # 权限声明（空格分隔）
        "exp": now + datetime.timedelta(hours=TOKEN_EXPIRE_HOURS),  # 过期时间
        "iat": now,                                              # 签发时间
    }
    # jwt.encode 内部会做 Base64URL + HMAC 签名，返回 "xxx.yyy.zzz" 形式的字符串
    return jwt.encode(payload, SECRET_KEY, algorithm=ALGORITHM)


def verify_token(token: str) -> dict:
    """校验 JWT：验签 + 校验 exp/iat，返回 payload；不合法会抛 jwt 异常。"""
    return jwt.decode(token, SECRET_KEY, algorithms=[ALGORITHM])


# ================================================================
# 四、登录接口
# ================================================================
@app.post("/login")
def login(req: LoginRequest):
    """用户登录 → 认证服务查库 → 签发令牌

    课案原文走「返回错误体」的宽松写法：口令错也返回 HTTP 200，只在 body 里给 access_token=None。
    """
    user = USERS.get(req.username)
    if not user or user["password"] != req.password:
        return {"access_token": None, "token_type": None, "message": "用户名或密码错误"}

    token = create_token(req.username)
    return {"access_token": token, "token_type": "bearer", "message": "登录成功"}

下面是原脚本的「线程内自起 + 真登录自检」基础设施（`start_auth_service_in_thread` + `demo_login_flow`
+ `_cli_auth`）。**本 notebook 不调用**这些 —— 改用 §2.4 的 `subprocess.Popen` 起独立进程；
这里原样保留，只为让你看到「单文件自检」的原版写法长什么样。

In [ ]:
# ---------- 2.1（附）原脚本的线程内自检基础设施（仅对照，不调用） ----------
AUTH_HOST = "127.0.0.1"
AUTH_PORT = 8022          # 课案用的是 9000；这里避开常用端口，减少和本机其他服务撞车的概率
AUTH_URL = f"http://{AUTH_HOST}:{AUTH_PORT}"


def start_auth_service_in_thread(port: int = AUTH_PORT):
    """后台线程起 uvicorn，返回 (server, thread)；端口被占则返回 None（表示已有服务）。"""
    import socket

    sock = socket.socket()
    sock.settimeout(0.5)
    try:
        sock.connect((AUTH_HOST, port))
        print(f"ℹ️  {AUTH_HOST}:{port} 已有服务在运行，直接用它。")
        return None
    except OSError:
        pass
    finally:
        sock.close()

    import threading
    import time

    import uvicorn

    server = uvicorn.Server(
        uvicorn.Config(app, host=AUTH_HOST, port=port, log_level="warning")
    )
    thread = threading.Thread(target=server.run, daemon=True)
    thread.start()

    for _ in range(100):          # 最多等 10 秒
        if server.started:
            return server, thread
        time.sleep(0.1)
    print(f"❌ 认证服务启动失败：{AUTH_HOST}:{port} 无法监听。")
    return None


def demo_login_flow() -> None:
    """用 requests 真登录：Alice / Bob / 错误口令 三种情况各走一遍。"""
    import requests

    print(f"密钥来源：{'环境变量 ' + SECRET_KEY_ENV if SECRET_KEY_FROM_ENV else '内置演示默认值（生产请改用环境变量）'}")
    print(f"认证服务地址：{AUTH_URL}")
    print()

    for username, password in [("alice", "pass123"), ("bob", "pass456"), ("alice", "wrong")]:
        resp = requests.post(
            f"{AUTH_URL}/login",
            json={"username": username, "password": password},
            timeout=10,
        )
        data = resp.json()
        print(f"POST /login  {username}/{password}  → HTTP {resp.status_code}")
        print(f"   message      : {data.get('message')}")
        if data.get("access_token"):
            token = data["access_token"]
            # 令牌是敏感凭据：日志里只打前 30 个字符，绝不整条打印
            print(f"   access_token : {token[:30]}...（共 {len(token)} 字符，只展示前缀）")
            print(f"   payload 解出来: {verify_token(token)}")
        else:
            print(f"   access_token : None（登录失败，不签发令牌）")
        print()


def _cli_auth():
    started = start_auth_service_in_thread()
    try:
        demo_login_flow()
    finally:
        if started:
            server, thread = started
            server.should_exit = True
            thread.join(timeout=10)
            print("✅ 认证服务已关闭。")

    print()
    print("说明：本文件无参数运行时走的是「起服务 → 真登录 → 关服务」的自检流程。")
    print("      09_权限_服务端_jxsd.py 与 10_权限_客户端_jxsd.py 会用 importlib")
    print("      直接加载本模块，自己把认证服务跑起来，**不需要手动常驻**。")
    print("      只有想在浏览器/Postman 里手工调 /login 时，才需要把它单独跑着。")

### 2.2 MCP 服务端完整版：`JWTVerifier` + 两个工具（`09_权限_服务端_jxsd.py`）

服务端核心和原版几乎一样，只是 name 换成 `Protected API`、工具换成 `add` + `get_weather`。
关键点在 `verifier`：`required_scopes=["read"]` 让所有调用至少要带 `read` 权限，
而**验签 + 验时间（exp/iat）是 `JWTVerifier` 底层自动做的，服务端一行鉴权代码都不写**。

In [ ]:
# ---------- 2.2 服务端完整版核心：校验器 + 两个工具 ----------
import asyncio
import time

import threading

from fastmcp import FastMCP
from fastmcp.server.auth.providers.jwt import JWTVerifier

MCP_HOST = "127.0.0.1"
MCP_PORT = 8023
MCP_PATH = "/mcp"

verifier = JWTVerifier(
    public_key=SECRET_KEY,
    algorithm="HS256",
    required_scopes=["read"],       # 所有调用至少要有 read 权限
)

# name 会显示在客户端/Inspector 里；auth=verifier 就是挂上认证中间件
mcp = FastMCP(name="Protected API", auth=verifier)


@mcp.tool
def add(a: float, b: float) -> float:
    """两数相加"""
    # 注意这里没有任何鉴权代码 —— 请求能走到这一行，就说明令牌已经验过了
    return a + b


@mcp.tool
def get_weather(city: str) -> str:
    """查询指定城市的实时天气"""
    weather_map = {"上海": "晴 25 度", "北京": "多云 18 度"}
    return weather_map.get(city, f"{city} 天气未知")

服务端原脚本的「自检基础设施」：`load_sibling`（用 importlib 按文件名加载兄弟文件）、
`start_server_in_thread`（线程内起服务端）、`self_check`（无令牌 / 合法令牌 / 伪造令牌三种情况）。
**本 notebook 不调用**这些（`Path(__file__)` 在 notebook 里不存在，且 §2.4 改用 Popen）；原样保留作对照。

In [ ]:
# ---------- 2.2（附）服务端原脚本的自检基础设施（仅对照，不调用） ----------
import importlib.util
from pathlib import Path
import socket


def load_sibling(filename: str, module_name: str):
    """从同目录按文件名加载模块（文件名以数字开头，没法直接 import）。"""
    if module_name in sys.modules:
        return sys.modules[module_name]

    path = Path(__file__).resolve().parent / filename
    spec = importlib.util.spec_from_file_location(module_name, path)
    module = importlib.util.module_from_spec(spec)
    sys.modules[module_name] = module
    spec.loader.exec_module(module)
    return module


# 原脚本在模块顶层就做这两行（import 时执行）拿共享密钥；notebook 里 SECRET_KEY 已在 §2.1 定义好，
# 这两行只作「原样对照」，放进一个永不调用的函数里，避免真去按文件路径加载兄弟文件。
def _load_shared_secret_reference():
    auth_service = load_sibling("08_权限_认证服务_jxsd.py", "mcp_auth_service_08")
    SECRET_KEY = auth_service.SECRET_KEY


def start_server_in_thread():
    """后台线程起带认证的 MCP 服务端；端口被占则直接连已有的。"""
    sock = socket.socket()
    sock.settimeout(0.5)
    try:
        sock.connect((MCP_HOST, MCP_PORT))
        print(f"ℹ️  {MCP_HOST}:{MCP_PORT} 已有服务在运行，直接连它。")
        return None
    except OSError:
        pass
    finally:
        sock.close()

    import uvicorn

    app = mcp.http_app(transport="streamable-http", path=MCP_PATH)
    server = uvicorn.Server(
        uvicorn.Config(app, host=MCP_HOST, port=MCP_PORT, log_level="warning")
    )
    thread = threading.Thread(target=server.run, daemon=True)
    thread.start()
    for _ in range(100):
        if server.started:
            return server, thread
        time.sleep(0.1)
    print(f"❌ MCP 服务端启动失败：{MCP_HOST}:{MCP_PORT} 无法监听。")
    return None


async def self_check(url: str) -> None:
    """三种情况各走一遍：无令牌 / 合法令牌 / 伪造令牌。"""
    from fastmcp import Client
    from fastmcp.client.auth import BearerAuth
    from fastmcp.client.transports import StreamableHttpTransport

    print("① 不带令牌连接：")
    try:
        async with Client(url) as client:
            await client.list_tools()
        print("   ❌ 竟然通过了 —— 认证中间件没生效！")
    except Exception as exc:
        print(f"   ✅ 已被拒绝：{type(exc).__name__}（HTTP 401 Unauthorized）")

    print("\n② 用认证服务（08）签发的合法令牌连接：")
    token = auth_service.create_token("alice")
    transport = StreamableHttpTransport(url=url, auth=BearerAuth(token))
    async with Client(transport) as client:
        tools = await client.list_tools()
        print(f"   ✅ 通过，可用工具：{[t.name for t in tools]}")
        result = await client.call_tool("add", {"a": 3, "b": 5})
        print(f"   add(3, 5) = {result.content[0].text}")

    print("\n③ 用错误的密钥伪造令牌连接：")
    import jwt as pyjwt

    forged = pyjwt.encode(
        {
            "sub": "hacker",
            "scope": "read execute",
            "exp": int(time.time()) + 3600,
            "iat": int(time.time()),
        },
        "wrong-secret-key-that-attacker-guessed",
        algorithm="HS256",
    )
    try:
        transport = StreamableHttpTransport(url=url, auth=BearerAuth(forged))
        async with Client(transport) as client:
            await client.list_tools()
        print("   ❌ 伪造令牌竟然通过了 —— 密钥配置有问题！")
    except Exception as exc:
        print(f"   ✅ 已被拒绝：{type(exc).__name__}（签名校验失败）")


def _cli_server():
    started = start_server_in_thread()
    url = f"http://{MCP_HOST}:{MCP_PORT}{MCP_PATH}"
    print(f"带认证的 MCP 服务端：{url}（required_scopes=['read']）\n")

    try:
        asyncio.run(self_check(url))
    finally:
        if started:
            server, thread = started
            server.should_exit = True
            thread.join(timeout=10)
            print("\n✅ 本文件启动的 MCP 服务端已关闭。")

    print()
    print("配套：10_权限_客户端_jxsd.py 演示「登录 → 拿令牌 → 调用」的完整链路。")

### 2.3 客户端完整版：`login` + `try_connect` + 一个正例三个反例（`10_权限_客户端_jxsd.py`）

客户端核心就两个函数：

- `login()`：向认证服务 `POST /login` 换令牌。注意**成功与否看字段不看状态码**——
  认证服务走「宽松错误」写法，口令错也返回 200，只是 `access_token` 为 `None`；
- `try_connect()`：拿令牌连一次 MCP 服务端，成功打印工具清单 + `add(3,5)`，被拒打印异常类型。

`main()` 用它俩跑「**一个正例 + 三个反例**」，把「签发权 / 校验权 / 持有权」的分工演示清楚。

In [ ]:
# ---------- 2.3 客户端完整版核心：login / try_connect / main ----------
import asyncio
import time

AUTH_HOST = "127.0.0.1"
AUTH_PORT = 8022               # 与 08_权限_认证服务_jxsd.py 一致
MCP_HOST = "127.0.0.1"
MCP_PORT = 8023                # 与 09_权限_服务端_jxsd.py 一致
MCP_PATH = "/mcp"
AUTH_URL = f"http://{AUTH_HOST}:{AUTH_PORT}"
MCP_URL = f"http://{MCP_HOST}:{MCP_PORT}{MCP_PATH}"


def login(username: str, password: str) -> str | None:
    """① 向认证服务登录，换回 JWT（课案里 TOKEN = "eyJ..." 那一行的真实来源）。"""
    import requests

    # 08 的 /login 走「宽松错误」写法：口令错也返回 200，只是 access_token 为 None，
    # 所以成功与否必须看字段，不能看 HTTP 状态码。
    resp = requests.post(
        f"{AUTH_URL}/login",
        json={"username": username, "password": password},
        timeout=10,
    )
    data = resp.json()
    if not data.get("access_token"):
        print(f"   登录失败（{username}）：{data.get('message')}")
        return None
    return data["access_token"]


async def try_connect(label: str, token: str | None) -> None:
    """用给定令牌连一次 MCP 服务端，打印成功/被拒的结论。"""
    from fastmcp import Client
    from fastmcp.client.auth import BearerAuth
    from fastmcp.client.transports import StreamableHttpTransport

    if token is None:
        transport = StreamableHttpTransport(url=MCP_URL)
    else:
        transport = StreamableHttpTransport(url=MCP_URL, auth=BearerAuth(token))

    try:
        async with Client(transport) as client:
            tools = await client.list_tools()
            print(f"   ✅ {label}：通过，可用工具 {[t.name for t in tools]}")
            result = await client.call_tool("add", {"a": 3, "b": 5})
            print(f"      add(3, 5) = {result.content[0].text}")
    except Exception as exc:
        print(f"   ⛔ {label}：被拒（{type(exc).__name__}: {str(exc).splitlines()[0][:90]}）")


async def main() -> None:
    # notebook 里用一个命名空间对象代替「importlib 按文件路径加载兄弟模块」——
    # create_token / verify_token / SECRET_KEY / ALGORITHM 都在 §2.1 定义好了。
    auth_service = _auth_namespace()
    print("=" * 64)
    print("② 向认证服务登录，拿 JWT")
    print("=" * 64)
    token = login("alice", "pass123")
    if not token:
        print("❌ 拿不到令牌，后续演示无法继续。")
        return
    print(f"   拿到 JWT：{token[:40]}...（共 {len(token)} 字符）")
    print(f"   payload：{auth_service.verify_token(token)}")
    print()
    print("=" * 64)
    print("③④ 携带 Bearer Token 调用 MCP 服务端")
    print("=" * 64)
    await try_connect("合法令牌（alice, scope='read execute'）", token)

    print()
    print("=" * 64)
    print("反例一：口令错误，认证服务不签发令牌")
    print("=" * 64)
    bad_token = login("alice", "wrong-password")
    await try_connect("匿名/无令牌", bad_token)

    print()
    print("=" * 64)
    print("反例二：令牌合法但没有 read 权限（required_scopes=['read']）")
    print("=" * 64)
    no_scope_token = auth_service.create_token("bob", scopes="")
    await try_connect("无 scope 令牌", no_scope_token)

    print()
    print("=" * 64)
    print("反例三：令牌已过期（exp 在过去）")
    print("=" * 64)
    import jwt as pyjwt

    expired = pyjwt.encode(
        {
            "sub": "alice",
            "scope": "read execute",
            "exp": int(time.time()) - 60,      # 60 秒前就过期了
            "iat": int(time.time()) - 3600,    # 签发时间放在 1 小时前，时间顺序自洽
        },
        auth_service.SECRET_KEY,
        algorithm=auth_service.ALGORITHM,
    )
    await try_connect("已过期令牌", expired)
    print("   ↑ 课案那句话的实证：JWTVerifier 底层用 pyjwt 自动校验 exp，服务端无需写代码。")

`main()` 里用到的 `_auth_namespace()` 就是把 §2.1 定义的四个名字打包成一个「模块等价物」——
原脚本里这些是通过 `load_sibling` 从 `08_权限_认证服务_jxsd.py` 拿的；notebook 里不加载兄弟文件，
直接复用 §2.1 已经定义好的函数与常量。

In [ ]:
import types


def _auth_namespace():
    """打包 §2.1 的签发/校验函数与密钥，等价于原脚本里的 auth_service 模块。"""
    return types.SimpleNamespace(
        SECRET_KEY=SECRET_KEY,
        ALGORITHM=ALGORITHM,
        create_token=create_token,
        verify_token=verify_token,
    )

客户端原脚本的「自检基础设施」：`load_sibling`（带注释版）+ `port_open` + `start_uvicorn_in_thread`
+ `_cli_client`。**本 notebook 不调用**这些 —— §2.4 改用 `subprocess.Popen`；原样保留作对照。

In [ ]:
# ---------- 2.3（附）客户端原脚本的自检基础设施（仅对照，不调用） ----------
import importlib.util
from pathlib import Path
import socket


def load_sibling(filename: str, module_name: str):
    """按文件名加载同目录模块（文件名以数字开头，无法直接 import）。"""
    if module_name in sys.modules:
        return sys.modules[module_name]

    path = Path(__file__).resolve().parent / filename
    spec = importlib.util.spec_from_file_location(module_name, path)
    module = importlib.util.module_from_spec(spec)
    sys.modules[module_name] = module      # 先注册再 exec，否则模块内的自引用会找不到自己
    spec.loader.exec_module(module)
    return module


def port_open(host: str, port: int) -> bool:
    """能连上就说明端口已被监听。用短超时，别让探测本身把脚本拖慢。"""
    sock = socket.socket()
    sock.settimeout(0.5)
    try:
        sock.connect((host, port))
        return True
    except OSError:
        return False
    finally:
        sock.close()


def start_uvicorn_in_thread(app, host: str, port: int):
    """通用：把任意 ASGI 应用跑在后台线程，返回 (server, thread)；已占用则返回 None。"""
    if port_open(host, port):
        print(f"ℹ️  {host}:{port} 已有服务在运行，直接连它。")
        return None                        # None = 「不是我起的」，调用方据此决定要不要关它

    import uvicorn

    server = uvicorn.Server(
        uvicorn.Config(app, host=host, port=port, log_level="warning")
    )
    thread = threading.Thread(target=server.run, daemon=True)   # daemon：主线程退出即收摊
    thread.start()
    for _ in range(100):                   # 最多轮询 10 秒（100 × 0.1s）
        if server.started:
            return server, thread
        time.sleep(0.1)
    print(f"❌ {host}:{port} 启动失败。")
    return None


def _cli_client():
    auth_module = load_sibling("08_权限_认证服务_jxsd.py", "mcp_auth_service_08")
    server_module = load_sibling("09_权限_服务端_jxsd.py", "mcp_protected_server_09")

    auth_started = start_uvicorn_in_thread(auth_module.app, AUTH_HOST, AUTH_PORT)
    mcp_app = server_module.mcp.http_app(transport="streamable-http", path=MCP_PATH)
    mcp_started = start_uvicorn_in_thread(mcp_app, MCP_HOST, MCP_PORT)

    print(f"认证服务：{AUTH_URL}      MCP 服务端：{MCP_URL}\n")

    try:
        asyncio.run(main())
    finally:
        for started, name in ((mcp_started, "MCP 服务端"), (auth_started, "认证服务")):
            if started:                     # None 表示那是外面已经在跑的服务，绝不能动它
                server, thread = started
                server.should_exit = True
                thread.join(timeout=10)
                print(f"✅ {name}已关闭。")

### 2.4 起服务：把两个服务落盘成独立脚本 + `subprocess.Popen` 后台起 + 语义校验

现在把 §2.1 / §2.2 的核心**落盘成两个能独立跑的脚本**，再用 `subprocess.Popen` 各起一个独立进程。
端口改成 notebook 专属的 **9110 / 8140**（源脚本的 8022 / 8023 只作对照）。

两个脚本共享**同一把密钥**（HS256 对称签名）——认证服务用它签发，服务端用它验签，
这就是课案说的「内部微服务共享同一密钥」。

In [ ]:
# ---------- 2.4 起服务：落盘脚本 + Popen + 轮询 + 语义校验 ----------
import subprocess
import threading

AUTH_DIR = WORKDIR / "mcp_jwt_auth"       # 本课专属子目录（WORKDIR 是同章共享的，必须再套一层）
AUTH_DIR.mkdir(parents=True, exist_ok=True)

AUTH_PORT = NB_AUTH_PORT                   # 8022 → 9110
MCP_PORT = NB_MCP_PORT                     # 8023 → 8140
AUTH_URL = f"http://{AUTH_HOST}:{AUTH_PORT}"
MCP_URL = f"http://{MCP_HOST}:{MCP_PORT}{MCP_PATH}"
print("认证服务地址：", AUTH_URL)
print("MCP 服务端地址：", MCP_URL)

AUTH_SCRIPT = AUTH_DIR / "auth_service.py"
MCP_SCRIPT = AUTH_DIR / "mcp_server.py"

AUTH_SOURCE = '''# -*- coding: utf-8 -*-
"""MCP 认证服务（由 notebook 04_权限_JWT认证.ipynb 生成）：签发 JWT 的 /login 接口。"""
import datetime
import os

import jwt
from fastapi import FastAPI
from pydantic import BaseModel

SECRET_KEY_ENV = "MCP_JWT_SECRET"
SECRET_KEY = os.environ.get(SECRET_KEY_ENV) or "demo-shared-secret-key-minimum-32-chars"
ALGORITHM = "HS256"
TOKEN_EXPIRE_HOURS = 1

app = FastAPI(title="MCP 认证服务")

USERS = {
    "alice": {"password": "pass123"},
    "bob":   {"password": "pass456"},
}


class LoginRequest(BaseModel):
    username: str
    password: str


def create_token(user_id: str, scopes: str = "read execute") -> str:
    now = datetime.datetime.now(datetime.timezone.utc)
    payload = {
        "sub": user_id,
        "scope": scopes,
        "exp": now + datetime.timedelta(hours=TOKEN_EXPIRE_HOURS),
        "iat": now,
    }
    return jwt.encode(payload, SECRET_KEY, algorithm=ALGORITHM)


def verify_token(token: str) -> dict:
    return jwt.decode(token, SECRET_KEY, algorithms=[ALGORITHM])


@app.post("/login")
def login(req: LoginRequest):
    user = USERS.get(req.username)
    if not user or user["password"] != req.password:
        return {"access_token": None, "token_type": None, "message": "用户名或密码错误"}
    token = create_token(req.username)
    return {"access_token": token, "token_type": "bearer", "message": "登录成功"}


if __name__ == "__main__":
    import uvicorn

    uvicorn.run(app, host="127.0.0.1", port=9110, log_level="warning")
'''

MCP_SOURCE = '''# -*- coding: utf-8 -*-
"""受保护的 MCP 服务端（由 notebook 04_权限_JWT认证.ipynb 生成）：JWTVerifier 校验 Bearer Token。"""
import os

from fastmcp import FastMCP
from fastmcp.server.auth.providers.jwt import JWTVerifier

SECRET_KEY = os.environ.get("MCP_JWT_SECRET") or "demo-shared-secret-key-minimum-32-chars"

verifier = JWTVerifier(
    public_key=SECRET_KEY,
    algorithm="HS256",
    required_scopes=["read"],       # 所有调用至少要有 read 权限
)

mcp = FastMCP(name="Protected API", auth=verifier)


@mcp.tool
def add(a: float, b: float) -> float:
    """两数相加"""
    return a + b


@mcp.tool
def get_weather(city: str) -> str:
    """查询指定城市的实时天气"""
    weather_map = {"上海": "晴 25 度", "北京": "多云 18 度"}
    return weather_map.get(city, f"{city} 天气未知")


if __name__ == "__main__":
    import uvicorn

    app = mcp.http_app(transport="streamable-http", path="/mcp")
    uvicorn.run(app, host="127.0.0.1", port=8140, log_level="warning")
'''

AUTH_SCRIPT.write_text(AUTH_SOURCE, encoding="utf-8")
MCP_SCRIPT.write_text(MCP_SOURCE, encoding="utf-8")
print("已落盘：", AUTH_SCRIPT.name, "与", MCP_SCRIPT.name, "（", AUTH_DIR.name, "目录）")

起服务用 `subprocess.Popen`（独立进程），并用一个「**语义校验**」确认起来的**真是我们的服务**：
光「端口有人监听」不够——监听的可能不是我们的服务（本机别的进程也可能占着）。所以等端口打开后，
再真发一个请求：认证服务看 `/login` 是否返回 `access_token`，MCP 服务端看 `list_tools` 是否列出 `add` / `get_weather`。

> 注意一个「原来如此」：MCP 服务端是**受保护的**，`list_tools` 不带令牌会被 401 拦下——
> 所以语义校验本身就得先 `create_token("alice")` 签个合法令牌带上。**这恰好就是认证中间件生效的现场证据**。

In [ ]:
import requests


def auth_semantic_ok() -> bool:
    """语义校验认证服务：真 POST /login，看 alice/pass123 是否签出 access_token。"""
    try:
        r = requests.post(f"{AUTH_URL}/login",
                          json={"username": "alice", "password": "pass123"}, timeout=10)
        return bool(r.json().get("access_token"))
    except Exception:
        return False


async def _mcp_list_tools():
    from fastmcp import Client
    from fastmcp.client.auth import BearerAuth
    from fastmcp.client.transports import StreamableHttpTransport

    # 语义校验也要带合法令牌：服务端是「受保护的」，不带令牌连 list_tools 都会 401
    # （这本身恰好证明认证中间件生效了）。用 §2.1 的 create_token 签一个合法令牌。
    token = create_token("alice")
    transport = StreamableHttpTransport(url=MCP_URL, auth=BearerAuth(token))
    async with Client(transport) as client:
        return [t.name for t in await client.list_tools()]


def run_async(coro):
    """在独立线程里 asyncio.run(coro)，异常回抛主线程，正常则返回协程结果。

    ipykernel 主线程已经有运行中的事件循环，顶层 asyncio.run() 会抛
    RuntimeError: asyncio.run() cannot be called from a running event loop；
    在子线程里跑就没有这个问题。
    """
    box = {}

    def _target():
        try:
            box["value"] = asyncio.run(coro)
        except BaseException as exc:      # noqa: BLE001
            box["error"] = exc

    t = threading.Thread(target=_target, daemon=True)
    t.start()
    t.join()
    if "error" in box:
        raise box["error"]
    return box.get("value")

先起**认证服务**（9110），再起 **MCP 服务端**（8140）。顺序不能反：服务端验签用的密钥和认证服务是同一把，
认证服务必须先可用（虽然本课两个都用同一个演示密钥硬编码，逻辑上仍是「先有签发方，再有校验方」）。

两个进程句柄存进 `_auth_proc` / `_mcp_proc`，最后一个 cell 用 `taskkill /F /T` 连子进程树一起收。

In [ ]:
# ---------- 起认证服务（9110） ----------
_auth_proc = None
_mcp_proc = None
_auth_ready = False
_mcp_ready = False

_env = {**os.environ, "PYTHONUTF8": "1", "PYTHONIOENCODING": "utf-8",
        "NO_PROXY": "127.0.0.1,localhost"}
_auth_log = open(AUTH_DIR / "auth_service.log", "w", encoding="utf-8")
_auth_proc = subprocess.Popen(
    [sys.executable, str(AUTH_SCRIPT)],
    cwd=str(ROOT), env=_env,
    stdout=_auth_log, stderr=subprocess.STDOUT,
)

for _ in range(60):                       # 最多等 30 秒
    if _auth_proc.poll() is not None:
        break
    if _port_in_use("127.0.0.1", NB_AUTH_PORT):
        _auth_ready = True
        break
    time.sleep(0.5)

if _auth_ready and auth_semantic_ok():
    print(f"✅ 认证服务已启动：{AUTH_URL}（已确认 /login 能签出 access_token）")
else:
    _auth_ready = False
    print(f"❌ 认证服务启动失败（日志见 {AUTH_DIR / 'auth_service.log'}）")

### 预期输出

```text
✅ 认证服务已启动：http://127.0.0.1:9110（已确认 /login 能签出 access_token）
```

语义校验不是「端口有人监听」就完事，而是真的 `POST /login` 拿到了 `access_token` ——
这样才不会把本机别的服务误认成认证服务。

再起 **MCP 服务端**（8140），同样轮询 + 语义校验（`list_tools` 要列出 `add` / `get_weather`）。

In [ ]:
# ---------- 起 MCP 服务端（8140） ----------
_mcp_log = open(AUTH_DIR / "mcp_server.log", "w", encoding="utf-8")
_mcp_proc = subprocess.Popen(
    [sys.executable, str(MCP_SCRIPT)],
    cwd=str(ROOT), env=_env,
    stdout=_mcp_log, stderr=subprocess.STDOUT,
)

for _ in range(60):                       # 最多等 30 秒
    if _mcp_proc.poll() is not None:
        break
    if _port_in_use("127.0.0.1", NB_MCP_PORT):
        _mcp_ready = True
        break
    time.sleep(0.5)

if _mcp_ready:
    try:
        _probe_names = run_async(_mcp_list_tools())
        if "add" in _probe_names and "get_weather" in _probe_names:
            print(f"✅ MCP 服务端已启动：{MCP_URL}（已确认工具 {_probe_names}）")
        else:
            _mcp_ready = False
            print(f"❌ 端口 {NB_MCP_PORT} 上监听的不是本课服务端（工具清单：{_probe_names}）")
    except Exception as exc:              # noqa: BLE001
        _mcp_ready = False
        print(f"❌ MCP 服务端就绪语义校验失败：{type(exc).__name__}: {exc}")
else:
    print(f"❌ MCP 服务端启动失败（日志见 {AUTH_DIR / 'mcp_server.log'}）")

### 预期输出

```text
✅ MCP 服务端已启动：http://127.0.0.1:8140/mcp（已确认工具 ['add', 'get_weather']）
```

工具顺序 = 服务端注册顺序：先 `add` 再 `get_weather`（和 `mcp_server.py` 里的 `@mcp.tool` 顺序一致）。

### 2.5 跑完整链路：一个正例 + 三个反例

两个服务都起来了，现在跑 §2.3 的 `main()`。它会把「谁说了算」演示清楚：

| 用例 | 令牌 | 结果 | 撞的是哪道关 |
|---|---|---|---|
| 正例 | alice 合法令牌（`scope="read execute"`） | ✅ 通过，能 `add` / `get_weather` | — |
| 反例一 | 口令错 → `access_token=None` | ⛔ 被拒 | **认证服务**（签发关） |
| 反例二 | 令牌合法但 `scope=""` | ⛔ 被拒 | **服务端** `required_scopes=["read"]` |
| 反例三 | 令牌已过期（`exp` 在过去） | ⛔ 被拒 | **JWTVerifier 自动校验时间** |

用 `run_async()` 把协程丢进新线程跑，绕开「内核已有事件循环」的坑（见「常见坑」第 2 条）。

In [ ]:
# ---------- 2.5 跑完整链路（正例 + 三个反例） ----------
if _auth_ready and _mcp_ready:
    run_async(main())
else:
    print("[跳过] 服务未就绪，跳过完整链路演示（请看上面起服务那两格的报错）。")

### 预期输出

```text
================================================================
② 向认证服务登录，拿 JWT
================================================================
   拿到 JWT：eyJhbGciOiJIUzI1NiIsInR5cCI6IkpXVCJ9.eyJ...（共 177 字符，这里每次运行都不同）
   payload：{'sub': 'alice', 'scope': 'read execute', 'exp': 1789640000, 'iat': 1789636400}

================================================================
③④ 携带 Bearer Token 调用 MCP 服务端
================================================================
   ✅ 合法令牌（alice, scope='read execute'）：通过，可用工具 ['add', 'get_weather']
      add(3, 5) = 8.0

================================================================
反例一：口令错误，认证服务不签发令牌
================================================================
   登录失败（alice）：用户名或密码错误
   ⛔ 匿名/无令牌：被拒（HTTPStatusError: Client error '401 Unauthorized' ...）

================================================================
反例二：令牌合法但没有 read 权限（required_scopes=['read']）
================================================================
   ⛔ 无 scope 令牌：被拒（HTTPStatusError: Client error '401 Unauthorized' ...）

================================================================
反例三：令牌已过期（exp 在过去）
================================================================
   ⛔ 已过期令牌：被拒（HTTPStatusError: Client error '401 Unauthorized' ...）
   ↑ 课案那句话的实证：JWTVerifier 底层用 pyjwt 自动校验 exp，服务端无需写代码。
```

> ⚠️ **本格输出不确定**：JWT 的 `payload` 里 `iat` / `exp` 是**时间戳**，令牌前缀后的部分是
> **每次运行都不同**，`177` 与那串数字只是本次实跑的示意，**别逐字比对**。真正要确认的只有四条：

1. **正例**：能列出 `['add', 'get_weather']` 且 `add(3, 5) = 8.0` —— 合法令牌放行；
2. **反例一**：口令错 → 认证服务返回 `access_token=None` → 客户端等同匿名 → 被 401 拦下；
3. **反例二**：令牌签名合法但 `scope=""` → 过不了 `required_scopes=["read"]` → 被拒。
   ⚠️ 本机 FastMCP 3.4.7 实测它也是 **401**（不是课案注释里写的 403）——「验签/验时间/验 scope」
   三道关都挂在认证中间件里，服务端统一回 401；
4. **反例三**：用**正确密钥**签的过期令牌 → 签名能过、时间过不了 → 被拒（**401**）——
   这一条最能说明「服务端一行鉴权代码没写，时间校验是 JWTVerifier 自动做的」。

## 小结

- **JWT 是 `header.payload.signature`**：payload 只是 Base64 编码（不是加密），
  安全性来自「改一个字签名就对不上」——所以**绝不能放口令、手机号**进 JWT；
- **三个角色互相制衡**：签发权在认证服务、校验权在服务端、持有权在客户端——
  客户端改一个字符，签名就对不上，所以它**无法自己给自己加权限**；
- **`auth=JWTVerifier(...)` 就是「认证中间件」**：无令牌 / 坏令牌直接 401，
  **根本进不到工具函数**，业务代码一行鉴权都不用写；
- **`required_scopes=["read"]`** 靠 payload 里的 `scope` 声明（**空格分隔字符串**）做权限判断——
  **令牌里必须有 `scope`**，否则所有调用都 401；
- **验签 + 验时间（exp/iat）是 JWTVerifier 自动做的**：过期令牌签名再对也 401，服务端无需写代码；
- **带认证的服务只能 HTTP 运行**：stdio 没有 HTTP 请求头，塞不进 `Authorization`。

## 常见坑

1. **`create_token` 只签 `sub/exp/iat`、没签 `scope` 时，配了 `required_scopes=["read"]` 会全部 401**：
   JWTVerifier 靠 `scope` 声明做权限判断，没有它只能「验签名」，区分不了 read / write。
2. **`asyncio.run()` 在 notebook 里会抛 `RuntimeError`**：ipykernel 主线程已经有运行中的事件循环，
   必须把协程丢进**新线程**再 `asyncio.run()`（见 `run_async`）；顶层 `await` 也会被判语法错误。
3. **`uvicorn.run()` / `mcp.run()` 会永久阻塞内核**：常驻服务必须落盘后 `subprocess.Popen` 起子进程，
   末尾 `taskkill /F /T` 连子进程树一起收。
4. **不要在进程内 import 归档区脚本**：归档脚本开头的 `sys.stdout.reconfigure()` 在 IPython 里会
   `AttributeError`（`OutStream` 没有 `reconfigure`）；要用服务端就 `Popen` 起独立脚本。
5. **就绪判断别只用 `connect_ex`**：它只要端口有人监听就返回 True，可能把本机别的服务误认成自己的。
   要再真发一个请求做**语义校验**（`/login` 返回 `access_token`、`list_tools` 列出自己的工具）。
6. **端口别用 8000/9000/8022/8023**：同章 5 个 notebook 并发，各用各的高位端口（本课 9110 / 8140）。
7. **`Path(__file__)` 在 notebook 里不存在**：一律改成 `WORKDIR` 下已落盘的脚本路径。
8. **成功与否看字段不看状态码**：认证服务走「宽松错误」写法，口令错也返回 200，只是 `access_token=None`。
9. **JWT 是敏感值**：日志里只打前缀（`token[:30]` / `token[:40]`），别整条打印、别粘进文档。

## 官方链接

- FastMCP Auth（`JWTVerifier` / `BearerAuth`）：<https://gofastmcp.com/deployment/auth>
- MCP 规范 · 授权（OAuth / JWT）：<https://modelcontextprotocol.io/specification/2025-06-18/basic/authorization>
- PyJWT（`jwt.encode` / `jwt.decode`）：<https://pyjwt.readthedocs.io/>
- FastAPI（`FastAPI` / `BaseModel` / `@app.post`）：<https://fastapi.tiangolo.com/>

## 最后：关掉两个服务

模板第 6 节第 5 条要求：常驻服务必须在**最后一个 cell** 里关掉。两个服务**都要收**，
Windows 上要**连子进程树一起收**（`/T`）——只杀父进程会留下孤儿继续占端口。

顺带演示 `WinError 32` 的应对：刚被 `taskkill` 的进程文件句柄不会立刻释放，
删临时目录要**重试 + 容忍失败**，不能因为删不掉就报错。

In [ ]:
# ---------- 收尾：关掉两个服务（连子进程树一起收） + 清理临时目录 ----------
for _proc, _name in ((_auth_proc, "认证服务"), (_mcp_proc, "MCP 服务端")):
    if _proc is not None and _proc.poll() is None:
        subprocess.run(["taskkill", "/F", "/T", "/PID", str(_proc.pid)],
                       stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL, check=False)
        print(f"✅ 已关闭 {_name}（PID {_proc.pid}）。")
    else:
        print(f"（{_name} 本就未启动或已退出，无需关闭）")

# 关掉日志文件句柄
for _log in (_auth_log, _mcp_log):
    if _log is not None and not _log.closed:
        _log.close()

# 等端口释放，再复查一次
for _ in range(20):
    if not _port_in_use("127.0.0.1", NB_AUTH_PORT) and not _port_in_use("127.0.0.1", NB_MCP_PORT):
        break
    time.sleep(0.25)
print("端口已释放：", not _port_in_use("127.0.0.1", NB_AUTH_PORT) and not _port_in_use("127.0.0.1", NB_MCP_PORT))


def remove_temp_dir(path):
    """删临时目录：Windows 上刚被 taskkill 的进程句柄未释放会 WinError 32，重试 + 容忍失败。"""
    import shutil

    for _ in range(5):
        try:
            shutil.rmtree(path)
            return True
        except FileNotFoundError:
            return True
        except OSError:
            time.sleep(0.5)
    return False


print("临时目录已清理：", remove_temp_dir(AUTH_DIR))